#Método de Uniformización para Cadenas de Markov en Tiempo Continuo (CTMC)
A continuación se realizarán las funciones necesarias para concluir la actividad 6.

In [14]:
import sympy as sp
import math

Generamos la matriz de tasas de cambio $R$

In [15]:
R = sp.Matrix([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
])

R

Matrix([
[0, 2, 3, 0],
[4, 0, 2, 0],
[0, 2, 0, 2],
[1, 0, 3, 0]])

Creamos los racionales $a = \frac{1}{6} , b = \frac{1}{3} , c = \frac{1}{2}$

In [16]:
a = sp.Rational(1,6)
b = sp.Rational(1,3)
c = sp.Rational(1,2)

Calculamos la matriz $\hat{P} = [\hat{p_{i,j}}]$ como:
$$ \hat{p_{i,j}} = f(x)=
\begin{cases}
1-\frac{r_i}{r} & \text{si } i=j,\\\\
\frac{r_{i,j}}{r} & \text{si } i\ne j.
\end{cases}  $$
donde $r$ es tal que $r \geq max{\{r_i\}}, (1 \leq i \leq N)$

In [17]:
#En este caso tomamos R = 6
Phat = sp.Matrix([
    [a, b, c, 0],
    [2*b, 0, b, 0],
    [0, b, b, b],
    [a, 0, c, b]
])
Phat

Matrix([
[1/6, 1/3, 1/2,   0],
[2/3,   0, 1/3,   0],
[  0, 1/3, 1/3, 1/3],
[1/6,   0, 1/2, 1/3]])

Para agilizar los cálculos obtendremos una aproximación numérica de la matriz $\hat{P}$ (no simbólica).

In [18]:
Phat2 = Phat.evalf()
Phat2

Matrix([
[0.166666666666667, 0.333333333333333,               0.5,                 0],
[0.666666666666667,                 0, 0.333333333333333,                 0],
[                0, 0.333333333333333, 0.333333333333333, 0.333333333333333],
[0.166666666666667,                 0,               0.5, 0.333333333333333]])

A continuación definimos la función para aproximar $P(t)$ usando los primeros $M$ términos de la serie, haciendo uso del $Teorema$ $3.3$:
<br>
La matriz de probabilidades de transición
$$
P(t)=\bigl[p_{i,j}(t)\bigr]
$$
está dada por
$$
P(t)=\sum_{k=0}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}\,\hat{P}^{\,k}.
$$

In [19]:
def aproximacion(t, r):
    #Tomamos M siguiendo la sugerencia de la actividad
    M = max(math.ceil(r*t + 5*math.sqrt(r*t)), 20)

    #Iniciamos Pt como una matriz 4x4 con ceros
    Pt = sp.zeros(4)

    #Calculamos el i-ésimo término de la serie y lo sumamos a Pt
    for k in range(M + 1):
        coef = sp.exp(-r*t) * (r*t)**k / sp.factorial(k)
        Pt += coef * (Phat2**k)

    #Devolvemos la evaluación numérica de Pt
    return Pt.evalf()

In [20]:
#Calculamos P(0.5), P(1) y P(5), y los mostramos
P05 = aproximacion(0.5, 6)
P1 = aproximacion(1, 6)
P5 = aproximacion(5, 6)

print("P(0.5)=")
sp.pprint(P05.evalf(8))

print("\nP(1)=")
sp.pprint(P1.evalf(8))

print("\nP(5)=")
sp.pprint(P5.evalf(8))

P(0.5)=
⎡0.25060868  0.2169646   0.38665694  0.14576979⎤
⎢                                              ⎥
⎢0.25313484  0.23836098  0.37440924  0.13409493⎥
⎢                                              ⎥
⎢0.1691195   0.19361489  0.42030102  0.2169646 ⎥
⎢                                              ⎥
⎣0.15801748  0.15744464  0.39833179  0.28620609⎦

P(1)=
⎡0.20615112  0.20390203  0.3987096   0.1912358 ⎤
⎢                                              ⎥
⎢0.20828421  0.2053407   0.39789917  0.18847446⎥
⎢                                              ⎥
⎢0.19675849  0.19837934  0.40095869  0.20390203⎥
⎢                                              ⎥
⎣0.19204622  0.19399715  0.40147094  0.21248423⎦

P(5)=
⎡0.19999963  0.19999963  0.39999925  0.19999962⎤
⎢                                              ⎥
⎢0.19999963  0.19999963  0.39999925  0.19999962⎥
⎢                                              ⎥
⎢0.19999962  0.19999962  0.39999925  0.19999963⎥
⎢                                              

Ahora verificaremos que se cumplan las ecuaciones de Chapman-Kolmogorov. Recordemos que, para una cadena de Markov en tiempo continuo, dichas ecuaciones establecen que

$$
P(s+t)=P(s)P(t), \qquad s,t\geq 0,
$$

donde $P(t)$ es la matriz de probabilidades de transición en el tiempo $t$.

En particular, para $s=t=0.5$ se tiene
$$
P(1)=P(0.5)P(0.5).
$$

A continuación, compararemos ambas matrices y calcularemos su diferencia para verificar numéricamente que esta igualdad se cumple.

In [21]:
print("Verificando las ecuaciones de Chapman-Kolmogorov \n")
CK = P05 * P05

print("P(0.5)*P(0.5) =\n")
sp.pprint(CK.evalf(8))

print("\nDiferencia ( P(0.5)P(0.5) - P(1) ):\n")
sp.pprint((P1 - CK).evalf(8))

print("\nVerificamos que en efecto la diferencia es muy pequeña. Por lo tanto las ecuaciones de Chapman-Kolmogorov se cumplen.")

Verificando las ecuaciones de Chapman-Kolmogorov 

P(0.5)*P(0.5) =

⎡0.20615141  0.20390232  0.39871018  0.19123609⎤
⎢                                              ⎥
⎢0.20828451  0.20534099  0.39789976  0.18847475⎥
⎢                                              ⎥
⎢0.19675878  0.19837963  0.40095927  0.20390232⎥
⎢                                              ⎥
⎣0.19204651  0.19399744  0.40147152  0.21248452⎦

Diferencia ( P(0.5)P(0.5) - P(1) ):

⎡-2.9101668e-7  -2.9101668e-7  -5.8203336e-7  -2.9101668e-7⎤
⎢                                                          ⎥
⎢-2.9101668e-7  -2.9101668e-7  -5.8203336e-7  -2.9101668e-7⎥
⎢                                                          ⎥
⎢-2.9101668e-7  -2.9101668e-7  -5.8203336e-7  -2.9101668e-7⎥
⎢                                                          ⎥
⎣-2.9101668e-7  -2.9101668e-7  -5.8203336e-7  -2.9101668e-7⎦

Verificamos que en efecto la diferencia es muy pequeña. Por lo tanto las ecuaciones de Chapman-Kolmogorov se cumplen.


<br>
<br>
Ahora definiremos la función correspondiente al ejercicio que hace uso del $Teorema$ $4$:

Para un $t\ge 0$ fijo, sea
<br>
$$
P^{M}(t)=\bigl[p^{M}_{i,j}(t)\bigr]
=\sum_{k=0}^{M}
e^{-rt}\frac{(rt)^k}{k!}\,\hat P^{\,k}.
$$

Entonces,

$$
\left|p_{i,j}(t)-p^{M}_{i,j}(t)\right|
\le
\sum_{k=M+1}^{\infty}
e^{-rt}\frac{(rt)^k}{k!},
\qquad
1\le i,j\le N.
$$
<br>
La matriz de tasas es la misma $R$ de antes.

In [22]:
R

Matrix([
[0, 2, 3, 0],
[4, 0, 2, 0],
[0, 2, 0, 2],
[1, 0, 3, 0]])

Definimos la función que aproxima $P(t)$ con tolerancia $\epsilon$. Siguiendo el algoritmo dado en el ejercicio.

In [23]:
def uniformizacion(R,t,eps):

    # n es el tamaño de R
    n = R.rows

    # ri es el vector con las sumas de las filas de R
    ri = []
    for i in range (n):
        ri.append(sum(R.row(i)))

    # r es tal que r >= max{r_i} 1 <= i <= N
    r = max(ri)

    # matriz P circunflejo
    Phat = sp.zeros(n)

    # construímos P cicunflejo según la definción
    for i in range(n):
        for j in range(n):
            if i==j:
                Phat[i,j] = 1-ri[i]/r
            else:
                Phat[i,j] = R[i,j]/r

    A = Phat
    c = sp.exp(-r*t)
    B = c*sp.eye(n)
    suma = c
    k = 1

    while suma < 1-eps:
        c = c*(r*t)/k
        B = B + c*A
        A = A*Phat
        suma += c
        k += 1

    M = k-1

    return B,M

In [24]:
#Calculamos P(0.5), P(1) y P(5), y los mostramos
P05,M05 = uniformizacion(R,0.5,1e-5)
P1,M1 = uniformizacion(R,1,1e-5)
P5,M5 = uniformizacion(R,5,1e-5)
print("P(0.5): ")
sp.pprint(P05.evalf())
print("M(0.5) =",M05)

print("\nP(1): ")
sp.pprint(P1.evalf())
print("M(1) =",M1)

print("\nP(5): ")
sp.pprint(P5.evalf())
print("M(5) =",M5)

P(0.5): 
⎡0.25060799926361   0.216963917777424  0.386655574821384  0.145769106222969⎤
⎢                                                                          ⎥
⎢0.253134164463168  0.238360303398857  0.37440787895663   0.134094251266731⎥
⎢                                                                          ⎥
⎢0.169118816135443  0.19361420786495   0.420299656307569  0.216963917777424⎥
⎢                                                                          ⎥
⎣0.158016802087722  0.157443961179206  0.398330429777621  0.286205405040838⎦
M(0.5) = 13

P(1): 
⎡0.206150375144699  0.203901281681478  0.398708105671623  0.191235057333263⎤
⎢                                                                          ⎥
⎢0.208283469478015  0.205339954005043  0.397897684531967  0.188473711816038⎥
⎢                                                                          ⎥
⎢0.196757748367714  0.198378590647027  0.400957199134844  0.203901281681478⎥
⎢                                              

Ahora verificaremos que se cumplan las ecuaciones de Chapman--Kolmogorov. Recordemos que, para una cadena de Markov en tiempo continuo, dichas ecuaciones establecen que

$$
P(s+t)=P(s)P(t), \qquad s,t\geq 0,
$$

donde $P(t)$ es la matriz de probabilidades de transición en el tiempo $t$.

En particular, para $s=t=0.5$ se tiene
$$
P(1)=P(0.5)P(0.5).
$$

A continuación, compararemos ambas matrices y calcularemos su diferencia para verificar numéricamente que esta igualdad se cumple.

In [25]:
print("Verificando las ecuaciones de Chapman-Kolmogorov\n")

P05, M05 = uniformizacion(R, 0.5, 1e-5)
P1, M1 = uniformizacion(R, 1, 1e-5)

CK = P05*P05

print("P(0.5)P(0.5) =\n")
sp.pprint(CK.evalf(8))

print("\nP(1) =\n")
sp.pprint(P1.evalf(8))

print("\nDiferencia ( P(1) - P(0.5)P(0.5) ):\n")
sp.pprint((P1 - CK).evalf(8))

print("\nVerificamos que, en efecto, la diferencia es muy pequeña.")
print("Por lo tanto, las ecuaciones de Chapman-Kolmogorov se cumplen.")

Verificando las ecuaciones de Chapman-Kolmogorov

P(0.5)P(0.5) =

⎡0.20615005  0.20390096  0.39870746  0.19123473⎤
⎢                                              ⎥
⎢0.20828314  0.20533963  0.39789703  0.18847339⎥
⎢                                              ⎥
⎢0.19675742  0.19837827  0.40095655  0.20390096⎥
⎢                                              ⎥
⎣0.19204515  0.19399608  0.4014688   0.21248316⎦

P(1) =

⎡0.20615038  0.20390128  0.39870811  0.19123506⎤
⎢                                              ⎥
⎢0.20828347  0.20533995  0.39789768  0.18847371⎥
⎢                                              ⎥
⎢0.19675775  0.19837859  0.4009572   0.20390128⎥
⎢                                              ⎥
⎣0.19204548  0.1939964   0.40146945  0.21248349⎦

Diferencia ( P(1) - P(0.5)P(0.5) ):

⎡3.2473005e-7  3.2472995e-7  6.4945942e-7  3.2472931e-7⎤
⎢                                                      ⎥
⎢3.2473013e-7  3.2473003e-7  6.494594e-7   3.2472915e-7⎥
⎢                             

# Comparación de resultados
Las matrices obtenidas con ambos métodos son prácticamente iguales. Las diferencias numéricas son muy pequeñas, por lo que la aproximación basada en la tolerancia $\epsilon = 10^{-5}$ produce esencialmente los mismos resultados que la aproximación usando
$$
M \approx \max\left\{rt + 5\sqrt{rt},\,20\right\}.
$$